# Example 0.13.3 / 3: Exercise 17-2: Soft Demand Limits with Piecewise-Linear Penalties

Companion notebook for **0.13.3 Exercises on Special Structures** in the AMPL lab Documentation, section "Exercise 17-2: Soft Demand Limits with Piecewise-Linear Penalties". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Exercise 17-2: Soft Demand Limits with Piecewise-Linear Penalties

Exercise 17-2 replaces the fixed demands of the book's transportation model by four limits per
destination: absolute and preferred minimums and maximums. The amount received, `Receive[j]`, must
stay within the absolute limits, and each unit outside the preferred range costs a penalty, 2 below
and 4 above. That penalty is a three-piece linear function of `Receive[j]` with slopes $-2, 0, 4$
and breakpoints at the preferred limits (Piecewise-Linear Terms (AMPL lab documentation, section 10.4),
Soft Constraints and Penalties (AMPL lab documentation, section 10.5)). Writing it as `<<...>> (Receive[j], dem_min_ask[j])`
makes the penalty zero at the preferred minimum, so within the preferred range it adds nothing.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ORIG;                                   # mills
set DEST;                                   # factories
param supply {ORIG} >= 0;                   # tons available
param cost {ORIG, DEST} >= 0;               # dollars per ton shipped
param dem_min_abs {DEST} >= 0;              # tons that must be received
param dem_min_ask {j in DEST} >= dem_min_abs[j];  # preferred minimum
param dem_max_ask {j in DEST} >= dem_min_ask[j];  # preferred maximum
param dem_max_abs {j in DEST} >= dem_max_ask[j];  # tons that may be received
param dem_min_pen >= 0;                     # dollars per ton below the preferred minimum
param dem_max_pen >= 0;                     # dollars per ton above the preferred maximum

var Trans {ORIG, DEST} >= 0;                # tons shipped
var Receive {j in DEST} >= dem_min_abs[j], <= dem_max_abs[j];  # tons received

minimize Total_Cost:
    sum {i in ORIG, j in DEST} cost[i,j] * Trans[i,j]                      # shipping
  + sum {j in DEST} <<dem_min_ask[j], dem_max_ask[j];
                     -dem_min_pen, 0, dem_max_pen>> (Receive[j], dem_min_ask[j]);  # penalties

subject to Supply {i in ORIG}: sum {j in DEST} Trans[i,j] = supply[i];    # ship all supply
subject to Demand {j in DEST}: sum {i in ORIG} Trans[i,j] = Receive[j];   # amount received

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
import pandas as pd

ampl.set["ORIG"] = ["GARY", "CLEV", "PITT"]
ampl.set["DEST"] = ["FRA", "DET", "LAN", "WIN", "STL", "FRE", "LAF"]
ampl.param["supply"] = {"GARY": 1400, "CLEV": 2600, "PITT": 2900}
ampl.param["cost"] = pd.DataFrame(
    [[39, 14, 11, 14, 16, 82, 8], [27, 9, 12, 9, 26, 95, 17], [24, 14, 17, 13, 28, 99, 20]],
    index=["GARY", "CLEV", "PITT"], columns=["FRA", "DET", "LAN", "WIN", "STL", "FRE", "LAF"],
)
limits = pd.DataFrame(
    {"dem_min_abs": [800, 975, 600, 350, 1200, 1100, 800],
     "dem_min_ask": [850, 1100, 600, 375, 1500, 1100, 900],
     "dem_max_ask": [950, 1225, 625, 450, 1800, 1100, 1050],
     "dem_max_abs": [1100, 1250, 625, 500, 2000, 1125, 1175]},
    index=["FRA", "DET", "LAN", "WIN", "STL", "FRE", "LAF"],
)
ampl.set_data(limits)
ampl.param["dem_min_pen"] = 2
ampl.param["dem_max_pen"] = 4

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result, ampl.obj["Total_Cost"].value())
received = ampl.var["Receive"].to_dict()
lo = ampl.param["dem_min_ask"].to_dict()
hi = ampl.param["dem_max_ask"].to_dict()
print("outside preferred range:", {j: r for j, r in received.items() if r < lo[j] or r > hi[j]})

HiGHS 1.14.0: optimal solution; objective 193075
12 simplex iterations
0 barrier iterations
solved 193075.0
outside preferred range: {'DET': 1250, 'LAF': 1175, 'STL': 1300, 'WIN': 500}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set ORIG := GARY CLEV PITT;
set DEST := FRA DET LAN WIN STL FRE LAF;
param supply := GARY 1400  CLEV 2600  PITT 2900;
param cost:  FRA  DET  LAN  WIN  STL  FRE  LAF :=
GARY          39   14   11   14   16   82    8
CLEV          27    9   12    9   26   95   17
PITT          24   14   17   13   28   99   20;
param:  dem_min_abs  dem_min_ask  dem_max_ask  dem_max_abs :=
FRA         800          850          950         1100
DET         975         1100         1225         1250
LAN         600          600          625          625
WIN         350          375          450          500
STL        1200         1500         1800         2000
FRE        1100         1100         1100         1125
LAF         800          900         1050         1175;
param dem_min_pen := 2;
param dem_max_pen := 4;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display {j in DEST: Receive[j] < dem_min_ask[j] or Receive[j] > dem_max_ask[j]} Receive[j];

HiGHS 1.14.0: optimal solution; objective 193075
12 simplex iterations
0 barrier iterations
Receive[j] [*] :=
DET  1250
LAF  1175
STL  1300
WIN   500
;

